
### Term Project 
#### Milestone 2

In [1]:
# libraries
import pandas as pd
import json
import ast
import re

In [2]:
# load the TMDb 5000 Movies data into a dataframe
movies = pd.read_csv('tmdb_5000_movies.csv')

# display the first few rows of data
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


### Step 1 - Financial Integrity
#### Identifies outliers and bad data

First I will focus on financial integrity of the data, since that is what I am focusing on for this project, I need to ensure that the data is accurate. Any empty financial records in budget or revenue will mean that I am not able to see the relationship between investment and success of a movie and therefore I should remove these \\$0 values to ensure my analysis will be accurate. These \\$0 budget and revenue columns would skew financial data visualizations. 

Later, when I join this spreadsheet with Box Office Mojo and OMDb API I want to ensure that I am only using movies that have an actual financial records.

In [3]:
# Step 1

# get the initial count of movies in the dataset
initial_count = len(movies)

# filter the dataframe to keep only rows with budget AND revenue > 0
movies_cleaned = movies[(movies['budget'] > 0) & (movies['revenue'] > 0)].copy()

final_count = len(movies_cleaned)
rows_removed = initial_count - final_count

print(f"*** Step #1 Complete ***")
print(f"Rows before: {initial_count}")
print(f"Rows after: {final_count}")
print(f"Total rows removed: {rows_removed}")

# show the first few rows of the updated financial data
print(movies_cleaned[['title', 'budget', 'revenue']].head())

*** Step #1 Complete ***
Rows before: 4803
Rows after: 3229
Total rows removed: 1574
                                      title     budget     revenue
0                                    Avatar  237000000  2787965087
1  Pirates of the Caribbean: At World's End  300000000   961000000
2                                   Spectre  245000000   880674609
3                     The Dark Knight Rises  250000000  1084939099
4                               John Carter  260000000   284139100


### Step 2 - JSON Strings to Readable Text
#### Format data into a more readable format

There are a number of columns that include JSON-formatted strings that have data that may be relevant during this analysis. For example, *genres*, contains data stored as a list of dictionaries in string format. To be able to use this for analysis and to be more easily readable I will extract only the primary genre (in cases of multiple genres listed) name from these strings and store it in a clean format. This means I will later be able to group movies by genre later in this analysis.

In [4]:
# function to get the primary genre name from the json when there is more than 1 genre listed
def extract_primary_genre(genre_str):
    try:
        # Convert the string to an actual list of dictionaries
        genres = ast.literal_eval(genre_str)
        if isinstance(genres, list) and len(genres) > 0:
            # Return the 'name' of the first genre in the list - assumption this is the 'primary' genre
            return genres[0]['name']
        return "Unknown"
    except (ValueError, SyntaxError):
        return "Unknown"

# apply the function to the genres column
movies_cleaned['genres'] = movies_cleaned['genres'].apply(extract_primary_genre)

print("*** Step #2 Complete ***")
print("Updated Genres Column (First 5 Rows):")
print(movies_cleaned[['title', 'genres']].head())

*** Step #2 Complete ***
Updated Genres Column (First 5 Rows):
                                      title     genres
0                                    Avatar     Action
1  Pirates of the Caribbean: At World's End  Adventure
2                                   Spectre     Action
3                     The Dark Knight Rises     Action
4                               John Carter     Action


### Step 3 - Date Features
#### Fix numerical values: string to datetime

The *release_date* column is stored as a string so I will need to convert this to a datetime format so I will be able to do time-series analysis or filter by years in my visualizations. I will also extract the *release_year* to its own column to enable easier data grouping.

In [5]:
# convert release_date to datetime (errors='coerce' handles any malformed strings)
movies_cleaned['release_date'] = pd.to_datetime(movies_cleaned['release_date'], errors='coerce')

# drop any rows where the date conversion failed (if any)
movies_cleaned = movies_cleaned.dropna(subset=['release_date'])

# extract the year and create a new column
movies_cleaned['release_year'] = movies_cleaned['release_date'].dt.year.astype(int)

print("*** Step #3 Complete ***")
print("Updated Date Features (First 5 Rows):")
print(movies_cleaned[['title', 'release_date', 'release_year']].head())

*** Step #3 Complete ***
Updated Date Features (First 5 Rows):
                                      title release_date  release_year
0                                    Avatar   2009-12-10          2009
1  Pirates of the Caribbean: At World's End   2007-05-19          2007
2                                   Spectre   2015-10-26          2015
3                     The Dark Knight Rises   2012-07-16          2012
4                               John Carter   2012-03-07          2012


### Step 4 - Column Pruning and Renaming
#### Replace Headers

I will remove columns that are empty or will be irrelevant to this analysis, like homepage, keywords, and tagline.  I will also rename the columns to follow a consistent, professional naming convention: PascalCase. I know later during this project I will need to create SQL tables and this will make my tables cleaner. It will also be important to make them clearer, like in the case of budget, I will name it *Budget_USD*, just in case I decide to pull any international box office data.

In [6]:
# define the columns I want to keep
cols_to_keep = [
    'title', 'genres', 'budget', 'revenue', 'release_year', 
    'popularity', 'runtime', 'vote_average', 'vote_count'
]

# prune the dataframe
movies_cleaned = movies_cleaned[cols_to_keep]

# rename headers for better readability
movies_cleaned = movies_cleaned.rename(columns={
    'title': 'Movie_Title',
    'genres': 'Primary_Genre',
    'budget': 'Budget_USD',
    'revenue': 'Revenue_USD',
    'release_year': 'Release_Year',
    'popularity': 'Popularity_Score',
    'runtime': 'Runtime_Minutes',
    'vote_average': 'Average_Rating',
    'vote_count': 'Total_Votes'
})

print("*** Step #4 Complete ***")
print("New Column Headers:")
print(movies_cleaned.columns.tolist())

*** Step #4 Complete ***
New Column Headers:
['Movie_Title', 'Primary_Genre', 'Budget_USD', 'Revenue_USD', 'Release_Year', 'Popularity_Score', 'Runtime_Minutes', 'Average_Rating', 'Total_Votes']


### Step 5 - Text Standardization
#### Normalize Data and Remove Duplicates

I want to ensure that there are no leading or trailing whitespaces anywhere that could cause an issue during the API call portion of this project. I will also check that there are no duplicates in the movie title column. I will also convert all titles to lowercase so I don't have to worry about them not matching later with the other data sources. While I do the title normalization I will also remove any random special characters that may prove to be an issue during later joining with Box Office Mojo and OMDb API.

In [7]:
# function to normalize the movie titles
def normalize_title(title):
    # convert to lowercase
    title = title.lower()
    # remove special characters (keep only letters and numbers)
    title = re.sub(r'[^a-z0-9\s]', '', title)
    # remove extra internal spaces
    title = re.sub(r'\s+', ' ', title).strip()
    return title

# apply normalization
# create a Simplified_Title column for the stripped version and use the pretty Movie_Title column for displays
movies_cleaned['Simplified_Title'] = movies_cleaned['Movie_Title'].apply(normalize_title)

print("*** Step #5 Complete ***")
print("Comparison of Original vs. Simplified Title:")
print(movies_cleaned[['Movie_Title', 'Simplified_Title']].head(5))

*** Step #5 Complete ***
Comparison of Original vs. Simplified Title:
                                Movie_Title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                         Simplified_Title  
0                                  avatar  
1  pirates of the caribbean at worlds end  
2                                 spectre  
3                   the dark knight rises  
4                             john carter  


### Ethical Implications of Data Wrangling: TMDB Movie Dataset

The dataset I used for this milestone was the TMDB 5000 Movies datset from kaggle, which is a community maintained subset of The Movie Database (TMDb). This is considered a credible source within the data science community, as Kaggle is widely used by data scientists. However, since it is a community contribution there is an inherent risk of data entry errors or missing financial records which I addressed as part of my data cleaning phase by removing rows that had \\$0 entries for budget or revenue which would skew my analysis. 
The main changes made to this dataset were filtering out records with \\$0 budget or revenus, parsing JSON strings to extract the primary genres, and normalizing the movie titles column. A significant assumption I made during this process was that a \\$0 value in the financial columns represented missing data rather than a literal \\$0 production (which seems unlikely to me). I also assumed that the first genre listed in the metadata array was the most accurate "primary" classification for each movie.

Since this data uses public movie metadata there are no real legal restrictions that I am worried about, like HIPAA if this was medical/personal data. However, I need to ensure that I respect the Terms of Service of the data providers. The primary regulatory concern would be to not misrepresent the financial health or success of specific production companies. 

The most significant ethical risk I believe that could have been created by my transformations is selection bias. Since I removed over 1,000 films that lacked budget or revenue data it is probable that the dataset leans more heavily toward mainstream Hollywood productions that actually report financial metrics. This means that any low budget, independent, student, or international films that may have high critic aclaim, but low financial transparency have been removed from this dataset. In an attempt to mitigate this risk I have kept the *Average_Rating* and *Total_Votes* columns in an attempt to ensure that success of a movie can still be measured by audience appreciation as well as total revenue. I will also ensure transparency in my final analysis that my data was limited to commerically reported movies. 

In [8]:
# length of the dataset after cleaning
print(f"The total length of the cleaned movies dataset is: {len(movies_cleaned)}")

# export to CSV
movies_cleaned.to_csv('movies_cleaned.csv')

# show the first few rows
movies_cleaned.head(10)

The total length of the cleaned movies dataset is: 3229


,Movie_Title,Primary_Genre,Budget_USD,Revenue_USD,Release_Year,Popularity_Score,Runtime_Minutes,Average_Rating,Total_Votes,Simplified_Title
0,Avatar,Action,237000000,2787965087,2009,150.437577,162.0,7.2,11800,avatar
1,Pirates of the Caribbean: At World's End,Adventure,300000000,961000000,2007,139.082615,169.0,6.9,4500,pirates of the caribbean at worlds end
2,Spectre,Action,245000000,880674609,2015,107.376788,148.0,6.3,4466,spectre
3,The Dark Knight Rises,Action,250000000,1084939099,2012,112.312950,165.0,7.6,9106,the dark knight rises
4,John Carter,Action,260000000,284139100,2012,43.926995,132.0,6.1,2124,john carter
5,Spider-Man 3,Fantasy,258000000,890871626,2007,115.699814,139.0,5.9,3576,spiderman 3
6,Tangled,Animation,260000000,591794936,2010,48.681969,100.0,7.4,3330,tangled
7,Avengers: Age of Ultron,Action,280000000,1405403694,2015,134.279229,141.0,7.3,6767,avengers age of ultron
8,Harry Potter and the Half-Blood Prince,Adventure,250000000,933959197,2009,98.885637,153.0,7.4,5293,harry potter and the halfblood prince
9,Batman v Superman: Dawn of Justice,Action,250000000,873260194,2016,155.790452,151.0,5.7,7004,batman v superman dawn of justice
